## 9b - Run Deprivation Model with AlphaEarth Satellite Embeddings

This script:
- Reads the per-buurt AlphaEarth embeddings (from pickle produced by script 9a)
- Selects which aggregation to use (mean, median, min, or max — all pre-computed in 9a)
- Joins with the CBS SES-WOA deprivation target
- Fits and evaluates a regression model (XGBoost) using cross-validation with hyperparameter tuning
- Tests out-of-sample performance on a held-out 20% test set (raw-score R²)
- Saves the best model for reuse

This mirrors the methodology of **script 4** (which uses street-level CLIP embeddings)
to enable a direct comparison between street-view and satellite-derived features for
predicting area-level deprivation.

### Dependencies
**Prerequisites:** Scripts 4 (best model for comparison) and 9a (AlphaEarth embeddings).

**Inputs:**
- `data/processed/per_buurt_embedding_summaries/alphaearth_embeddings.pkl` — from script 9a
- `data/models/best_model.joblib` — from script 4 (for comparison only)
- CBS SES-WOA target (via `amsterdam_data.get_ses_woa()`)
- Amsterdam buurt boundaries (via `amsterdam_data.get_amsterdam_buurten()`) — for residuals map
- `clustering_functions.py` — imports `RANDOM_STATE`, `embedding_statistic`

**Outputs:**
- `data/models/best_model_alphaearth.joblib` — best XGBoost pipeline for AlphaEarth embeddings

**Used by:** None (terminal analysis notebook)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*glibc.*")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import multiprocessing
import joblib
import geopandas as gpd


from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
from xgboost import XGBRegressor

from model_evaluation import compute_metrics


def evaluate_model(y_true, y_pred, column, num_in_class=None, num_areas=None,
                   plot=True, plot_collectively=False, ax=None):
    """Compute regression metrics (via compute_metrics) and optionally plot true vs predicted."""
    metrics = compute_metrics(y_true, y_pred)

    if plot:
        plt.figure(figsize=(6, 6))
        plt.scatter(y_true, y_pred, alpha=0.7)
        plt.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], 'r--', lw=2)
        plt.xlabel(f"True {column}")
        plt.ylabel(f"Predicted {column}")
        plt.title(f"{column}")
        plt.grid(True)
        plt.show()

    if plot_collectively and ax is not None:
        ax.scatter(y_true, y_pred, alpha=0.7)
        ax.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], 'r--', lw=2)
        ax.set_xlabel("True")
        ax.set_ylabel("Predicted")
        ax.set_title(f"{column}: \n{num_in_class} images\n {num_areas} buurten", fontsize=15)
        ax.grid(True)
        ax.text(
            0.01, 0.99,
            f'R² = {metrics["R2"]:.2f}, RMSE = {metrics["RMSE"]:.2f}',
            transform=ax.transAxes,
            fontsize=12,
            verticalalignment='top',
            horizontalalignment='left')

    return metrics

In [ ]:
from directory_filepaths import *
from clustering_functions import RANDOM_STATE, embedding_statistic
from amsterdam_data import get_ses_woa, get_amsterdam_buurten

In [ ]:
# --- Configuration ---

var_to_predict = TARGET_COL  # 'ses_woa_score' (continuous; PRIMARY)

# Which aggregation of the per-buurt AlphaEarth embeddings to use as features.
# Script 9a pre-computes all of these: 'mean', 'median', 'min', 'max'.
# We use the centrally-defined statistic from clustering_functions.py for consistency.
AGGREGATION = embedding_statistic

embedding_col = f'{AGGREGATION}_embedding'
print(f"Using '{embedding_col}' as features (from clustering_functions.py)")

### Get SES-WOA data

In [ ]:
ses = get_ses_woa()  # buurtcode, ses_woa_score, ses_welvaart, ses_opleiding, ses_arbeidsverleden
ses

### Sanity check: map SES-WOA score

Quick choropleth to verify the deprivation data looks sensible (lower scores = more deprived).

In [ ]:
buurten = get_amsterdam_buurten()  # EPSG:28992
buurten_ses = buurten.merge(ses, on=JOIN_KEY, how='left')

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
buurten_ses.plot(
    column='ses_woa_score',
    cmap='RdYlGn',  # red = low score (more deprived), green = high score (less deprived)
    legend=True,
    legend_kwds={'label': 'SES-WOA score (lower = more deprived)', 'shrink': 0.6},
    missing_kwds={'color': 'lightgrey'},
    ax=ax)
ax.set_title('CBS SES-WOA score by buurt - Amsterdam')
ax.set_axis_off()
plt.tight_layout()
plt.show()

### Get AlphaEarth embedding data

In [ ]:
# Load the AlphaEarth embeddings pickle (produced by script 9a).
# This contains multiple aggregation columns: mean_embedding, median_embedding, min_embedding, max_embedding.
embedding_pkl = os.path.join(data_dir, "per_buurt_embedding_summaries", "alphaearth_embeddings.pkl")
big_summary_df = pd.read_pickle(embedding_pkl)

print(f"Available aggregations: {[c for c in big_summary_df.columns if c.endswith('_embedding')]}")
print(f"Using: {embedding_col}")

big_summary_df_with_ses = pd.merge(left=big_summary_df, right=ses[[JOIN_KEY, 'ses_woa_score']], on=JOIN_KEY)

print(f"Buurten with both embeddings and SES-WOA data: {len(big_summary_df_with_ses)}")
print(f"Embedding dimensionality: {len(big_summary_df_with_ses.iloc[0][embedding_col])}")

### Split the data into 80% training, 20% testing

Same split parameters as script 4 (`RANDOM_STATE`) to ensure comparability.

In [ ]:
X = np.stack(big_summary_df_with_ses[embedding_col].values)
y = big_summary_df_with_ses[var_to_predict].values

# -------------------------
# Split data into training and test sets
# -------------------------
X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X, y, np.arange(X.shape[0]), test_size=0.2, random_state=RANDOM_STATE)
print(f"Training points: {X_train.shape[0]}, Test points: {X_test.shape[0]}")
print(f"Embedding dimensions: {X_train.shape[1]}")

### Perform model selection and hyper-parameter tuning using the 80% training data

Same hyperparameter grid and cross-validation setup as script 4.

In [ ]:
# Define model pipelines and parameter grids
# Each entry: (name, pipeline, param_grid)
model_configs = [
    ("XGBoost", Pipeline([
        ('scaler', StandardScaler()),
        ('reg', XGBRegressor(
            objective='reg:squarederror',  # minimise squared error (standard for regression)
            random_state=RANDOM_STATE,               # reproducible results
            n_jobs=-1,                     # use all CPU cores for training a single model
            verbosity=0))]),               # suppress XGBoost's own logging
     {
      'reg__n_estimators': [100, 300],         # number of boosting rounds (more = more complex)
      'reg__max_depth': [3, 6, 10],            # max tree depth (controls model complexity)
      'reg__learning_rate': [0.01, 0.1, 0.3],  # step size shrinkage (lower = slower but often better)
      'reg__subsample': [0.8, 1.0],            # fraction of training rows used per tree (< 1 adds regularisation)
     }),
]

# Cross-validation setup
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
ncores = min(multiprocessing.cpu_count() - 1, 100)

best_model = None
best_score = -np.inf
best_model_name = None
best_params = {}

print(f"Training {len(model_configs)} model(s) using {ncores} cores")

for name, pipeline, param_grid in model_configs:
    # Calculate fits for progress reporting
    n_fits = 1
    for vals in param_grid.values():
        n_fits *= len(vals)
    n_fits *= cv.n_splits
    print(f"\nTraining: {name} ({n_fits} fits)...")

    # Main grid search code:
    grid = GridSearchCV(pipeline, param_grid, cv=cv, scoring='r2', n_jobs=ncores, verbose=1)
    grid.fit(X_train, y_train)

    print(f"  CV mean R\u00b2 = {grid.best_score_:.3f}")
    print(f"  Best params: {grid.best_params_}")

    if grid.best_score_ > best_score:
        best_score = grid.best_score_
        best_model = grid.best_estimator_
        best_model_name = name
        best_params = grid.best_params_.copy()

print(f"\nBest model: {best_model_name} (CV R\u00b2 = {best_score:.3f})")
print(f"Best hyperparameters: {best_params}")

# Compute CV NRMSE for the best model (RMSE / std(y_test) per fold, then average)
# This uses the same approach as scripts 7 and 8 for comparability.
from sklearn.base import clone
nrmse_scores = []
for train_ix, val_ix in cv.split(X_train):
    fold_model = clone(best_model)
    fold_model.fit(X_train[train_ix], y_train[train_ix])
    y_val_pred = fold_model.predict(X_train[val_ix])
    fold_rmse = np.sqrt(mean_squared_error(y_train[val_ix], y_val_pred))
    nrmse_scores.append(fold_rmse / np.std(y_train[val_ix]))

cv_nrmse = np.mean(nrmse_scores)
print(f"CV mean NRMSE = {cv_nrmse:.3f}")

In [ ]:
# Save the best model
model_dir = os.path.join("../data/models")
os.makedirs(model_dir, exist_ok=True)

bundle = {
    "model": best_model,
    "name": best_model_name,
    "cv_score_r2": float(best_score),
    "cv_nrmse": float(cv_nrmse),
    "best_params": best_params,
    "embedding_source": "alphaearth",
    "embedding_dim": X_train.shape[1],
}

model_path = os.path.join(model_dir, "best_model_alphaearth.joblib")
joblib.dump(bundle, model_path)
print(f"Saved model to {model_path}")

## Test out-of-sample performance of 'best' model

Train on the full 80% training data, test on the 20% held-out set.

In [ ]:
# Predict on held-out test data
y_pred_test = best_model.predict(X_test)

# Evaluate using the function defined above
test_metrics = evaluate_model(
    y_pred=y_pred_test,
    y_true=y_test,
    column=var_to_predict,
    plot=True)

print("Test set metrics (AlphaEarth):")
print(f"  raw-score R² (PRIMARY): {test_metrics['R2']:.3f}")
for k, v in test_metrics.items():
    if k != 'R2':
        print(f"  {k}: {v:.3f}")

## Compare with street-view model

Load the street-view model from script 4 and compare performance side-by-side.

In [ ]:
# Load the street-view model results from script 4
sv_model_path = os.path.join("../data/models", "best_model.joblib")

if os.path.exists(sv_model_path):
    sv_bundle = joblib.load(sv_model_path)
    print("Street-view model (script 4):")
    print(f"  CV R\u00b2 = {sv_bundle['cv_score_r2']:.3f}")
    print(f"  Best params: {sv_bundle['best_params']}")
    print()
else:
    print(f"Street-view model not found at {sv_model_path}")
    print("Run script 4 first to enable comparison.")
    print()

# Compare with AlphaEarth
ae_bundle = joblib.load(os.path.join("../data/models", "best_model_alphaearth.joblib"))

print("="*60)
print("COMPARISON: Cross-validation metrics")
print("="*60)
if os.path.exists(sv_model_path):
    sv_nrmse = sv_bundle.get('cv_nrmse', 'N/A')
    sv_nrmse_str = f"{sv_nrmse:.3f}" if isinstance(sv_nrmse, float) else sv_nrmse
    print(f"  Street View (CLIP, 512-dim):      R\u00b2 = {sv_bundle['cv_score_r2']:.3f}, NRMSE = {sv_nrmse_str}")
print(f"  AlphaEarth  (satellite, 64-dim):  R\u00b2 = {ae_bundle['cv_score_r2']:.3f}, NRMSE = {ae_bundle['cv_nrmse']:.3f}")
print()
print("Test set metrics (AlphaEarth):")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.3f}")

Map of the residuals (true - predicted) for the AlphaEarth model across Amsterdam buurten.

In [ ]:
# Predict on ALL buurten (not just the test set) so we get a complete map
y_pred_all = best_model.predict(X)
residuals = y - y_pred_all  # positive = model under-predicted SES-WOA (true score higher)

# Attach residuals to the dataframe and merge with geometries
residual_df = big_summary_df_with_ses[[JOIN_KEY]].copy()
residual_df['residual'] = residuals

buurten = get_amsterdam_buurten()
ams_residuals = buurten.merge(residual_df, on=JOIN_KEY, how='inner')

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
vmax = np.percentile(np.abs(ams_residuals['residual']), 95)  # symmetric colour scale clipped at 95th percentile
ams_residuals.plot(
    column='residual',
    cmap='RdBu',       # red = negative residual (over-predicted SES-WOA score)
    vmin=-vmax,         # blue = positive residual (under-predicted SES-WOA score)
    vmax=vmax,
    legend=True,
    legend_kwds={'label': 'Residual (True − Predicted SES-WOA score)', 'shrink': 0.6},
    ax=ax)
ax.set_title('AlphaEarth Model Residuals — Amsterdam')
ax.set_axis_off()
plt.tight_layout()
plt.show()

print(f"Mapped {len(ams_residuals)} buurten")
print(f"Mean residual: {residuals.mean():.3f}, Std: {residuals.std():.3f}")